In [1]:
import numpy as np
import pandas as pd

from pathlib import Path
import os
os.environ['FUTSPREAD_ROOT'] = '/Users/robli/Documents/Columbia/2026Fall/DDF/data/futspread'

from futspread import build_combo, load_combo
from futspread.universe import POINT_VALUE

from src.backtest import *
from src.strategies import *

In [2]:
combos = build_combos()

In [3]:
rules = {'mean>2c': mean_vs_cost, 'hit70': hit_rate, 't>2': t_stat,
         'median': median_vs_cost, 'recent10': recent}
res = {name: summarise(backtest(combos, f, windows, '2006-01-01', '2020-12-31'))
       for name, f in rules.items()}
backtest_comparison = pd.DataFrame(res).T
backtest_comparison

,trades,won,gross,cost,net,net_per_trade
mean>2c,106.0,0.500000,105963.311534,52714.391871,53248.919663,502.348299
hit70,64.0,0.625000,331573.713012,31928.865376,299644.847636,4681.950744
t>2,49.0,0.632653,290437.207185,24446.122087,265991.085098,5428.389492
median,105.0,0.533333,157844.569270,52599.931591,105244.637680,1002.329883
recent10,110.0,0.509091,-60490.559330,54742.393997,-115232.953327,-1047.572303


I choose T_stat

In [9]:
final = backtest(combos, t_stat, windows, '2021-01-01', '2025-12-31')

trades_2021 = final[final['entry'].dt.year == 2021].copy()
trades_2021['direction'] = trades_2021['dir'].map({1: 'long', -1: 'short'})

# Past average in the trade's direction and at its actual size, in $
trades_2021['past_avg_pnl'] = trades_2021['dir'] * trades_2021['past_mean_usd'] * trades_2021['lots']

trades_2021 = trades_2021[['combo', 'direction', 'entry', 'exit', 'past_mean_usd', 'past_avg_pnl']]
trades_2021

,combo,direction,entry,exit,past_mean_usd,past_avg_pnl
5,HE[J][M0],short,2021-01-21,2021-02-18,-491.904781,7804.296213
15,ZW[N][Z0],short,2021-04-07,2021-05-05,-123.750000,2008.928571
10,ZS[N][X0],long,2021-04-07,2021-05-05,488.095238,3639.107087
21,CT[N][Z0],short,2021-04-29,2021-05-27,-845.476190,9630.117780
0,HO[V][F0],long,2021-08-05,2021-09-02,353.205800,2025.643584


In [11]:
RESULTS = Path('../results')
RESULTS.mkdir(exist_ok=True)

trades_2021.to_csv(RESULTS / 'trades_2021.csv')